# 01 — Exploratory Data Analysis
BTC/USDT OHLCV from Bitget: price, volume, returns

In [ ]:
import sys, os

sys.path.insert(0, "..")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from data.ingestion.rest_client import BitgetRESTClient
from data.ingestion.resampler import OHLCVResampler, Timeframe

%matplotlib inline
client = BitgetRESTClient(
    {
        "exchange": {
            "name": "bitget",
            "symbols": ["BTC/USDT"],
            "type": "spot",
            "rate_limit": {"max_requests_per_second": 10},
        },
        "data": {"validation": {"max_price_jump_pct": 30}},
    }
)
df_1h = client.fetch_days(timeframe="1h", days=90)
df_1h["dt"] = pd.to_datetime(df_1h["timestamp"], unit="ms")
df_1h = df_1h.set_index("dt")
df_4h = OHLCVResampler.resample_bulk(
    df_1h.reset_index().rename(columns={"dt": "timestamp"}), Timeframe.H4
)
df_1d = OHLCVResampler.resample_bulk(
    df_1h.reset_index().rename(columns={"dt": "timestamp"}), Timeframe.D1
)
print(f"{len(df_1h)} 1H bars | {len(df_4h)} 4H | {len(df_1d)} 1D")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
ax[0].plot(df_1h.index, df_1h["close"], lw=0.5, color="#00C9A7")
ax[0].set_title("BTC/USDT Close")
ax[1].bar(df_1h.index, df_1h["volume"], width=0.01, alpha=0.5)
ax[1].set_title("Volume")
r = df_1h["close"].pct_change().dropna()
ax[2].plot(r.index, r, lw=0.3, color="#FF6B6B")
ax[2].set_title("Returns")
plt.tight_layout()
plt.show()

In [ ]:
print(f"Close: ${df_1h.close.min():,.0f} - ${df_1h.close.max():,.0f}")
print(f"Vol (ann): {df_1h.close.pct_change().std() * np.sqrt(365 * 24) * 100:.1f}%")
print(
    f"Gaps >5%: {((df_1h.open - df_1h.close.shift(1)).abs() / df_1h.close.shift(1) > 0.05).sum()}"
)